In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps
# pylammpsmpi
sys.path.insert(0, '..')
from src import *

os.environ['OMP_NUM_THREADS'] = '10'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUDA_HOME'] = '/usr'

In [57]:
rho = 1.058  # Density (reduced units)
L = 10  # Box length (in lattice units)

epsilon = 1.0  # LJ energy scale
sigma = 1.0    # LJ length scale

offset = 0.05  # to avoid creating atoms on simbox face

T = 1.   # temperature (reduced units)
time_step = 0.002  # Time step (reduced units)
thermo_tau = 100 * time_step  # Thermostat damping parameter
# baro_tau = 1000 * time_step

save_every = 100
born_nl_save_every = 100

In [58]:
savedir = Filepaths.DATA.value / 'test'
os.makedirs(savedir, exist_ok=True)
print(savedir)

/home/fgarbuzov/Documents/lammps_test/data/test


## Init

In [59]:
#lmp = lammps(cmdargs=['-log', os.path.join(savedir,'sim.log'), '-sf', 'omp',
#                      "-screen", "none", "-nocite"])

In [60]:
13**3 / 1000 * 4000

8788.0

In [61]:
lmp = lammps(cmdargs=['-sf', 'omp', "-screen", "none", "-nocite"])

In [62]:
#L = 15

In [63]:
lmp.commands_string(f"""
# Initialization
units lj
dimension 3
boundary p p p
atom_style atomic

# Create simulation box and atoms
lattice fcc {rho} origin {offset} {offset} {offset}
region simbox block 0 {L} 0 {L} 0 {L} units lattice
create_box 1 simbox
create_atoms 1 box
mass 1 1.0
velocity all create {T} 94673 mom yes

# Interaction settings
pair_style lj/smooth 2.5 3.5
pair_coeff 1 1 1.0 1.0
pair_modify shift yes

# Neighbor list settings
# neighbor 0.3 bin
# neigh_modify every 1 delay 0 check yes 

# Temperature computation (unnecessary due to internal thermo_temp)
compute temp all temp

# Monitoring variables
variable time equal time
variable etot equal etotal
variable vol equal vol
variable dens equal density

# Pressure tensor computation (the vector has length 6)
compute press all pressure thermo_temp

# Born matrix computation
#compute press_pot all pressure NULL virial
#compute born_matrix all born/matrix numdiff 1.0e-6 press_pot
#compute born_matrix_nl all born/matrix/nonlinear 1.0e-6 press_pot born_matrix

# Screen and log output
thermo 5000
thermo_style custom step temp press density vol etotal
thermo_modify flush yes

# Timestep (default 0.005 for lennard-jones)
timestep {time_step}
""")

In [64]:
lmp.get_natoms()

4000

## Equilibration

In [13]:
# Equilibrate pressure
#fix npt_eq all npt temp {T} {T} {thermo_tau} iso 0.0 0.0 {baro_tau}
#run 5000
#unfix npt_eq

# Save equilibrated system
#write_data "{savedir}/equilibrated_T={T}.dump" nocoeff
# write_restart "{savedir}/equilibrated.restart"

In [71]:
lmp.commands_string(f"""
fix nvt_eq all nvt temp {T} {T} {thermo_tau}
run 100
unfix nvt_eq                    
""")

In [93]:
lmp.commands_string(f"""
fix npt_eq all npt temp {T} {T} {thermo_tau} iso 0.0 0.0 1.0
run 10000
unfix npt_eq                    
""")

In [94]:
lmp.numpy.extract_compute('press', 0, 1)

array([-0.01996234, -0.04587557, -0.01441537, -0.01471954, -0.01605113,
       -0.0432552 ])

In [95]:
lmp.get_thermo('vol')

6513.732341729046

In [96]:
lmp.get_thermo('ke')

1.5044675543214416

In [97]:
lmp.get_thermo('pe')

-3.9520016944083967

In [98]:
lmp.get_thermo('etotal')

-2.447534140086955

## Production

In [ ]:
# Save full system state every {sample_interval} steps
#dump snapshot_dump all custom {sample_interval} "{savedir}/snapshot_*.dat" &
#    id type x y z vx vy vz       

In [21]:
lmp.commands_string(f"""
fix nve_prod all nve

# Output the the desired values every {save_every} steps
fix thermo_output all ave/time 1 1 {save_every} &
   v_time c_thermo_temp v_etot v_vol v_dens file "{savedir}/thermo.dat"
fix press_output all ave/time 1 1 {save_every} &
   v_time c_press[*] file "{savedir}/press.dat"
fix press_pot_output all ave/time 1 1 {save_every} &
   v_time c_press_pot[*] file "{savedir}/press_pot.dat"
fix born_output all ave/time 1 1 {save_every} &
   v_time c_born_matrix[*] file "{savedir}/born.dat"
fix born_nl_output all ave/time 1 1 {born_nl_save_every} &
   v_time c_born_matrix_nl[*] file "{savedir}/born_nl.dat"          
""")

In [22]:
lmp.command(f"run 10000")

In [23]:
lmp.commands_string("""
unfix born_nl_output
""")

In [ ]:
lmp.command(f"run 10000000")

In [10]:
lmp.commands_string("""
#unfix thermo_output
#unfix press_output
unfix press_pot_output
unfix born_output
unfix born_nl_output
""")

In [ ]:
lmp.command(f"run 100000")

In [14]:
sys.path.insert(0, '..')
from src import *

def get_pk2_pot(lmp, cell_matrix_ref, F, eps=1e-7):
    set_cell_matrix(lmp, F@cell_matrix_ref, remap=True, convert=True)
    F_inv = np.linalg.inv(F)
    sigma = -lammps_array_to_matrix(lmp.numpy.extract_compute("press_pot", 0, 1))
    return np.linalg.det(F) * F_inv @ sigma @ F_inv.T

def compute_F_deform(E):
    return np.linalg.cholesky(np.eye(3) + 2*E, upper=True)

def compute_born_nonlin_pk2_der(lmp, cell_matrix=None, eps=1e-6, undeform=True):
    E_deforms, map_ij = probe_deformations_sym(eps)
    
    lmp.command("change_box all triclinic")
    h0 = get_cell_matrix(lmp) if cell_matrix is None else cell_matrix
    NB = np.zeros((3,)*6)
    
    for (k,l), dE1 in zip(map_ij, E_deforms):
        for (m,n), dE2 in zip(map_ij, E_deforms):
            S11 = get_pk2_pot(lmp, h0, compute_F_deform(- dE1 - dE2))
            S12 = get_pk2_pot(lmp, h0, compute_F_deform(- dE1 + dE2))
            S21 = get_pk2_pot(lmp, h0, compute_F_deform(+ dE1 - dE2))
            S22 = get_pk2_pot(lmp, h0, compute_F_deform(+ dE1 + dE2))
            NB[:,:,k,l,m,n] = (S22 + S11 - S12 - S21) / (4 * eps**2)
            if m != n:
                NB[:, :, k, l, n, m] = NB[:, :, k, l, m, n]
        if k != l:
            NB[:, :, l, k] = NB[:, :, k, l]
    for (i,j) in map_ij[2:]:
        NB[j,i] = NB[i,j]
    
    if undeform:
        set_cell_matrix(lmp, h0, remap=True)

    return NB

In [15]:
ntimes = 100
NB_arr = np.zeros((ntimes,) + (3,)*6)

for i in range(ntimes):
    NB_arr[i] = compute_born_nonlin_pk2_der(lmp, eps=1e-6, undeform=True)
    lmp.command(f"run 100")